## CONFIG AND SETUP

In [ ]:
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%idle_timeout 15
%region us-east-1
# %iam_role arn:aws:iam::958165011713:role/AWSGlueServiceRole-IcebergLab

In [ ]:
%%configure
{
  "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://iceberg-data-lake-958165011713/glue-iceberg-warehouse/ --conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog --conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO"
}

In [ ]:
spark
spark.sql("SELECT 1 AS test").show()

In [ ]:
spark.sql("SHOW DATABASES IN glue_catalog").show() #####checking our database is present or not

In [ ]:
#######################CHECK SPARK S3 CONNECTIVITY
spark.sql("""
CREATE TABLE IF NOT EXISTS glue_catalog.practice.spark_iceberg_test (
    id BIGINT,
    name STRING,
    city STRING,
    created_at TIMESTAMP
)
USING iceberg
""")

spark.sql("""
INSERT INTO glue_catalog.practice.spark_iceberg_test VALUES
(1, 'Ayush', 'Kolkata', current_timestamp()),
(2, 'Rahul', 'Delhi', current_timestamp()),
(3, 'Sneha', 'Bangalore', current_timestamp())
""")

spark.sql("""
SELECT *
FROM glue_catalog.practice.spark_iceberg_test
""").show()

## REAL WORK

In [ ]:
## Read already created dataset (Change your account name below)
orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("s3://iceberg-data-lake-958165011713/raw/500_k/orders_500k.csv")
)

In [ ]:
orders_df.printSchema()

In [ ]:
orders_df.show(5)

In [ ]:
#Register temp view:
orders_df.createOrReplaceTempView("orders_raw")
spark.sql("SHOW TABLES IN practice").show(truncate=False)

In [ ]:
spark.sql("""
CREATE TABLE glue_catalog.practice.orders_spark_month_city_customer_bucket
USING iceberg
PARTITIONED BY (
  months(order_date),
  city,
  bucket(10, customer_id)
)
AS
SELECT
  CAST(order_id AS BIGINT) AS order_id,
  CAST(customer_id AS STRING) AS customer_id,
  CAST(order_date AS DATE) AS order_date,
  product_category,
  city,
  CAST(order_amount AS DOUBLE) AS order_amount,
  order_status,
  payment_status,
  current_timestamp() AS updated_at
FROM orders_raw
WHERE customer_id IS NOT NULL
""")

In [ ]:
spark.sql("""
SELECT COUNT(*) AS total_partitions
FROM glue_catalog.practice.orders_spark_month_city_customer_bucket.partitions
""").show()